#GCN_PB IID (Having Isolated Nodes)

##Load Libaries

In [ ]:
!pip install ogb
!pip install torch_geometric

In [ ]:
pip install torch

##GCN Model Layers

In [ ]:
from torch._C import *
import torch
print(torch.__version__)

2.11.0+cu128


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINConv
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize

# GIN Model Implementation
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_classes, num_layers, dropout):
        super(GIN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.dropout = dropout

        # First layer
        mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )
        self.convs.append(GINConv(mlp1))

        # Intermediate layers
        for _ in range(num_layers - 2):
            mlp = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels)
            )
            self.convs.append(GINConv(mlp))

        # Final layer
        mlp_out = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )
        self.convs.append(GINConv(mlp_out))

        self.fc = nn.Linear(out_channels, num_classes)

    def reset_parameters(self):
        """Explicitly reset weights for re-training in K-Fold."""
        for conv in self.convs:
            # GINConv's internal MLP is accessed via conv.nn
            for layer in conv.nn:
                if hasattr(layer, 'reset_parameters'):
                    layer.reset_parameters()
        self.fc.reset_parameters()

    def forward(self, x, adj_t):
        for conv in self.convs:
            x = conv(x, adj_t)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.fc(x)
        return x

def train(model, data, train_idx, optimizer, weights=None):
    model.train()
    criterion = torch.nn.CrossEntropyLoss(weight=weights)
    optimizer.zero_grad()
    out = model(data.x, data.adj_t)[train_idx]
    loss = criterion(out, data.y[train_idx])
    loss.backward()
    optimizer.step()
    return loss.item()

def to_one_hot(y, num_classes):
    y_one_hot = torch.zeros(y.size(0), num_classes).to(y.device)
    y_one_hot.scatter_(1, y.view(-1, 1), 1)
    return y_one_hot

@torch.no_grad()
def test(model, data, split_idx, num_classes):
    model.eval()
    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    def compute_metrics(y_true_indices, probs):
        y_true_labels = y_true_indices.cpu().numpy()
        y_pred_labels = probs.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)

        # AUC Metrics
        y_true_bin = label_binarize(y_true_labels, classes=[0, 1])
        rocauc = roc_auc_score(y_true_bin, probs.cpu().numpy(), multi_class='ovr')
        precision, recall, _ = precision_recall_curve(y_true_bin[:, 1], probs[:, 1].cpu().numpy())
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(data.y[split_idx['train']], y_probs[split_idx['train']])
    valid_metrics = compute_metrics(data.y[split_idx['valid']], y_probs[split_idx['valid']])
    test_metrics = compute_metrics(data.y[split_idx['test']], y_probs[split_idx['test']])

    return train_metrics, valid_metrics, test_metrics

@torch.no_grad()
def test_kfold(model, data, split_idx, num_classes):
    model.eval()

    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    y_true_train = to_one_hot(data.y[split_idx['train']], num_classes)
    #y_true_valid = to_one_hot(data.y[split_idx['valid']], num_classes)
    y_true_test = to_one_hot(data.y[split_idx['test']], num_classes)

    def compute_metrics(y_true, y_probs):
        y_pred_labels = y_probs.argmax(dim=1).cpu().numpy()
        y_true_labels = y_true.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)
        rocauc = roc_auc_score(y_true.cpu(), y_probs.cpu())
        y_true_cpu = y_true.cpu()
        y_true_binarized = label_binarize(y_true_cpu, classes=[0, 1])
        y_probs_cpu = y_probs.cpu().numpy()
        precision, recall, _ = precision_recall_curve(y_true_binarized[:, 1], [score[1] for score in y_probs_cpu])
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(y_true_train, y_probs[split_idx['train']])
    #valid_metrics = compute_metrics(y_true_valid, y_probs[split_idx['valid']])
    test_metrics = compute_metrics(y_true_test, y_probs[split_idx['test']])

    return train_metrics, test_metrics

In [ ]:
'''import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.nn import GCNConv, SAGEConv
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,num_classes,num_layers, dropout):
        super(GAT, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.fc = torch.nn.ModuleList()
        self.convs.append(GATConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels, hidden_channels))
        self.convs.append(GATConv(hidden_channels, out_channels))
        self.fc.append(torch.nn.Linear(out_channels,num_classes))
        self.dropout = dropout


    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()

    def forward(self, x, adj_t):
        for conv in self.convs:
            x = conv(x, adj_t)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        #second to last layer i use this when i wanna apply dropout after fully connected layer
        for fc in self.fc[:-1]:
            x = fc(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.fc[-1](x)
        return x

def train(model, data, train_idx, optimizer):
      model.train()
      criterion = torch.nn.CrossEntropyLoss(weight=weights)
      optimizer.zero_grad()
      out = model(data.x, data.adj_t)[train_idx]
      loss = criterion(out, data.y[train_idx])
      loss.backward()
      optimizer.step()
      return loss.item()

def to_one_hot(y, num_classes):
    y_one_hot = torch.zeros(y.size(0), num_classes).to(y.device)
    y_one_hot.scatter_(1, y.view(-1, 1), 1)
    return y_one_hot

@torch.no_grad()
def test(model, data, split_idx, num_classes):
    model.eval()

    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    y_true_train = to_one_hot(data.y[split_idx['train']], num_classes)
    y_true_valid = to_one_hot(data.y[split_idx['valid']], num_classes)
    y_true_test = to_one_hot(data.y[split_idx['test']], num_classes)

    def compute_metrics(y_true, y_probs):
        y_pred_labels = y_probs.argmax(dim=1).cpu().numpy()
        y_true_labels = y_true.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)
        rocauc = roc_auc_score(y_true.cpu(), y_probs.cpu())
        #aucprc=0.6
        y_true_cpu = y_true.cpu()
        y_true_binarized = label_binarize(y_true_cpu, classes=[0, 1])
        y_probs_cpu = y_probs.cpu().numpy()
        precision, recall, _ = precision_recall_curve(y_true_binarized[:, 1], [score[1] for score in y_probs_cpu])
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(y_true_train, y_probs[split_idx['train']])
    valid_metrics = compute_metrics(y_true_valid, y_probs[split_idx['valid']])
    test_metrics = compute_metrics(y_true_test, y_probs[split_idx['test']])

    return train_metrics, valid_metrics, test_metrics

@torch.no_grad()
def test_kfold(model, data, split_idx, num_classes):
    model.eval()

    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    y_true_train = to_one_hot(data.y[split_idx['train']], num_classes)
    #y_true_valid = to_one_hot(data.y[split_idx['valid']], num_classes)
    y_true_test = to_one_hot(data.y[split_idx['test']], num_classes)

    def compute_metrics(y_true, y_probs):
        y_pred_labels = y_probs.argmax(dim=1).cpu().numpy()
        y_true_labels = y_true.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)
        rocauc = roc_auc_score(y_true.cpu(), y_probs.cpu())
        y_true_cpu = y_true.cpu()
        y_true_binarized = label_binarize(y_true_cpu, classes=[0, 1])
        y_probs_cpu = y_probs.cpu().numpy()
        precision, recall, _ = precision_recall_curve(y_true_binarized[:, 1], [score[1] for score in y_probs_cpu])
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(y_true_train, y_probs[split_idx['train']])
    #valid_metrics = compute_metrics(y_true_valid, y_probs[split_idx['valid']])
    test_metrics = compute_metrics(y_true_test, y_probs[split_idx['test']])

    return train_metrics, test_metrics'''

"import numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch_geometric.data import Data\nfrom torch_geometric.nn import GATConv\nfrom torch_geometric.nn import GCNConv, SAGEConv\nfrom sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score\nfrom sklearn.metrics import precision_recall_curve, auc\nfrom sklearn.preprocessing import label_binarize\nimport numpy as np\n\nclass GAT(torch.nn.Module):\n    def __init__(self, in_channels, hidden_channels, out_channels,num_classes,num_layers, dropout):\n        super(GAT, self).__init__()\n        self.convs = torch.nn.ModuleList()\n        self.fc = torch.nn.ModuleList()\n        self.convs.append(GATConv(in_channels, hidden_channels))\n        for _ in range(num_layers - 2):\n            self.convs.append(GATConv(hidden_channels, hidden_channels))\n        self.convs.append(GATConv(hidden_channels, out_channels))\n        self.fc.append(torch.nn.Linear(out_channels,num_classes))\n       

##Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import pandas as pd
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/embeddings_mean_protbert_3k.csv')
df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/protT5_embeddings.csv')
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/merged_df.csv')
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/New_embeddings_protbert_whole_nodes.csv')
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/New_embeddings_protbert_whole.csv')
#df_features = df_features.iloc[:19550,:]
#df_features.iloc[:,1:-1] = np.random.uniform(low=-3, high=6, size=(5609, 1024)) # replace with random

df_labels = df_features .iloc[:4819, [0, -1]]

In [ ]:
df_labels.shape

(4819, 2)

In [ ]:
# Convert both ID columns to strings (example)
df_labels['Id'] = df_labels['Id'].astype(str)
df_features['Id'] = df_features['Id'].astype(str)

In [ ]:
order_dict = {value: index for index, value in enumerate(list(df_labels['Id']))}

def sorting_key(value):
    return (order_dict.get(value, float('inf')), value)

df_features = df_features.sort_values(by='Id', key=lambda x: x.map(sorting_key))

In [ ]:
df_features.shape

(4819, 1026)

In [ ]:
import networkx as nx

graph_data = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/new_ppi_edges_iid.csv')
#graph_data = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/new_ppi_edges_iid_whole.csv')
#graph_data = pd.read_csv('/content/gdrive/MyDrive/Data/PPI-edges-STRING.csv')

id_list = list(df_features['Id'])

G = nx.Graph()
for id1,id2 in zip(list(graph_data.iloc[:,0]),list(graph_data.iloc[:,1])):
  if((str(id1) in id_list) and (str(id2) in id_list)):
    G.add_edge(id1,id2)

#Adding Isolated Nodes to graph which we have label for them and embeddings extracted from protbert.
for node in df_features.iloc[:, 0]:
    if node not in G.nodes():
        G.add_node(node)

In [ ]:
len(G.edges)

110907

In [ ]:
len(G.nodes())

4819

In [ ]:
df_features = df_features[df_features['Id'].isin(list(G.nodes()))]

In [ ]:
adj_sparse = nx.adjacency_matrix(G, nodelist=list(df_features.iloc[:,0]))

In [ ]:
node_labels = np.array(df_features.iloc[:,-1])

In [ ]:
from scipy.sparse import coo_matrix
adj_matrix = coo_matrix(adj_sparse).todense()

In [ ]:
import os
import numpy as np
import torch

# Define the number of nodes and features
num_nodes = 4819
num_features = 1024
num_classes = 2

node_features = np.array(df_features.iloc[:,1:-1])
node_names = np.array(df_features.iloc[:,0])

adj_matrix = (adj_matrix + adj_matrix.T) / 2
adj_matrix[adj_matrix < 0.9] = 0  # Sparsify the adjacency matrix

node_features_tensor = torch.from_numpy(node_features).float()
node_labels_tensor = torch.from_numpy(node_labels).long()  # Convert labels to long type

adj_coo = adj_sparse.tocoo()
indices = np.vstack((adj_coo.row, adj_coo.col))
values = adj_sparse.data

indices_tensor = torch.tensor(indices, dtype=torch.long)
values_tensor = torch.tensor(values, dtype=torch.float)

adj_t = torch.sparse_coo_tensor(indices_tensor, values_tensor, adj_sparse.shape).to_sparse_csr()

data = Data(uni_id=node_names ,x=node_features_tensor, adj_t=adj_t, y=node_labels_tensor)

# Assuming your data structure
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Map unique IDs to indices
id_to_idx_map = {id_: idx for idx, id_ in enumerate(data.uni_id)}

# Function to load folds (train.npy and test.npy) from Google Drive
def load_folds(folds_dir):
    folds = []
    for fold in range(1, 6):  # Assuming 5 folds (1 to 5)
        # Define the paths for train.npy and test.npy for each fold
        train_path = os.path.join(folds_dir, f'fold_{fold}_train_ids.csv')
        test_path = os.path.join(folds_dir, f'fold_{fold}_test_ids.csv')

        # Load the train and test files
        train_ids = pd.read_csv(train_path)  # This should load the numpy array of IDs
        test_ids = pd.read_csv(test_path)

        # Map the IDs to indices
        train_idx = torch.tensor([id_to_idx_map[id_] for id_ in list (train_ids.iloc[:,0])], dtype=torch.long).to(device)
        test_idx = torch.tensor([id_to_idx_map[id_] for id_ in list (test_ids.iloc[:,0])], dtype=torch.long).to(device)

        # Append the tuple of train and test indices for this fold
        folds.append((train_idx, test_idx))

    return folds

folds_dir = "/content/gdrive/MyDrive/New_Repo_Data/5folds"  # Update this path to your Google Drive directory
folds = load_folds(folds_dir)

/tmp/ipykernel_4844/4056065895.py:26: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj_t = torch.sparse_coo_tensor(indices_tensor, values_tensor, adj_sparse.shape).to_sparse_csr()
/tmp/ipykernel_4844/4056065895.py:26: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  adj_t = torch.sparse_coo_tensor(indices_tensor, values_tensor, adj_sparse.shape).to_sparse_csr()


##Run Model

In [ ]:
import os
import numpy as np
import torch
from sklearn.model_selection import KFold
import torch.nn.functional as F

# Hyper-parameters and settings
hidden_channels = 256
out_channels = 64
num_layers = 2
dropout = 0.5
runs = 5
lr = 0.001
epochs = 500
eval_steps = 10
log_steps = 10
weights = torch.tensor([2.0,1.0])
num_classes = 2
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load data and folds
data = data.to(device)
weights = weights.to(device)
model = GIN(in_channels=data.num_features,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            num_classes=num_classes,
            num_layers=num_layers,
            dropout=dropout).to(device)

# Metrics storage
train_acc_list, train_prec_list, train_rec_list = [], [], []
train_f1_list, train_rocauc_list, train_aucprc_list = [], [], []
test_acc_list, test_prec_list, test_rec_list = [], [], []
test_f1_list, test_rocauc_list, test_aucprc_list = [], [], []

# Main training loop
for fold, (train_idx, test_idx) in enumerate(folds):
    print(f'\n=== Fold {fold + 1}/{len(folds)} ===')
    train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
    test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)
    split_idx = {'train': train_idx, 'test': test_idx}

    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_train_rocauc = 0
    loss_history = []

    for epoch in range(1, epochs + 1):
        loss = train(model, data, train_idx, optimizer)
        loss_history.append(loss)

        # Early stopping check
        if len(loss_history) > 10 and np.std(loss_history[-10:]) < 1e-3:
            print(f"Early stopping at epoch {epoch}")
            break

        # Evaluation
        if epoch % eval_steps == 0 or epoch == epochs:
            train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)
            train_rocauc = train_metrics[4]

            # Save best model
            if train_rocauc > best_train_rocauc:
                best_train_rocauc = train_rocauc
                torch.save(model.state_dict(), f'best_model_fold_{fold}.pt')

    # Final evaluation with best model
    model.load_state_dict(torch.load(f'best_model_fold_{fold}.pt'))
    train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)

    # Store metrics
    for lst, values in zip([train_acc_list, train_prec_list, train_rec_list, train_f1_list, train_rocauc_list, train_aucprc_list],
                           train_metrics):
        lst.append(values)

    for lst, values in zip([test_acc_list, test_prec_list, test_rec_list, test_f1_list, test_rocauc_list, test_aucprc_list],
                           test_metrics):
        lst.append(values)

# Metrics calculations
def calculate_stats(metric_list, name):
    return {
        'mean': np.mean(metric_list),
        'std': np.std(metric_list),
        'var': np.var(metric_list),
        'min': np.min(metric_list),
        'max': np.max(metric_list)
    }

# Generate report
print("\n=== Final Report ===")
print("{:<15} {:<8} {:<8} {:<8} {:<8} {:<8}".format(
    'Metric', 'Mean', 'Std', 'Var', 'Min', 'Max'))

for metric_name, train_list, test_list in [
    ('Accuracy', train_acc_list, test_acc_list),
    ('Precision', train_prec_list, test_prec_list),
    ('Recall', train_rec_list, test_rec_list),
    ('F1', train_f1_list, test_f1_list),
    ('ROC AUC', train_rocauc_list, test_rocauc_list),
    ('PR AUC', train_aucprc_list, test_aucprc_list)
]:
    train_stats = calculate_stats(train_list, 'Train')
    test_stats = calculate_stats(test_list, 'Test')

    print(f"\n**{metric_name}**")
    print("Train:\t{mean:.4f} ± {std:.4f}\t(var: {var:.4f})\t[{min:.4f}-{max:.4f}]".format(**train_stats))
    print("Test:\t{mean:.4f} ± {std:.4f}\t(var: {var:.4f})\t[{min:.4f}-{max:.4f}]".format(**test_stats))


=== Fold 1/5 ===


/tmp/ipykernel_4844/2469250362.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
/tmp/ipykernel_4844/2469250362.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)



=== Fold 2/5 ===


/tmp/ipykernel_4844/2469250362.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
/tmp/ipykernel_4844/2469250362.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)



=== Fold 3/5 ===


/tmp/ipykernel_4844/2469250362.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
/tmp/ipykernel_4844/2469250362.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)



=== Fold 4/5 ===


/tmp/ipykernel_4844/2469250362.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
/tmp/ipykernel_4844/2469250362.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)



=== Fold 5/5 ===


/tmp/ipykernel_4844/2469250362.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
/tmp/ipykernel_4844/2469250362.py:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)



=== Final Report ===
Metric          Mean     Std      Var      Min      Max     

**Accuracy**
Train:	0.8595 ± 0.0055	(var: 0.0000)	[0.8542-0.8687]
Test:	0.8398 ± 0.0111	(var: 0.0001)	[0.8185-0.8496]

**Precision**
Train:	0.8237 ± 0.0112	(var: 0.0001)	[0.8143-0.8450]
Test:	0.8123 ± 0.0112	(var: 0.0001)	[0.7962-0.8295]

**Recall**
Train:	0.9933 ± 0.0103	(var: 0.0001)	[0.9732-1.0000]
Test:	0.9750 ± 0.0170	(var: 0.0003)	[0.9481-0.9919]

**F1**
Train:	0.9004 ± 0.0027	(var: 0.0000)	[0.8977-0.9046]
Test:	0.8861 ± 0.0079	(var: 0.0001)	[0.8716-0.8938]

**ROC AUC**
Train:	0.9311 ± 0.0054	(var: 0.0000)	[0.9225-0.9362]
Test:	0.8822 ± 0.0104	(var: 0.0001)	[0.8695-0.8941]

**PR AUC**
Train:	0.9582 ± 0.0033	(var: 0.0000)	[0.9530-0.9613]
Test:	0.9140 ± 0.0077	(var: 0.0001)	[0.9003-0.9215]


In [ ]:
for fold, (train_idx, test_idx) in enumerate(folds):
    if fold == 1:
       print(f'Fold {fold + 1}/{len(folds)}')

    # Convert indices to tensors and move to GPU/CPU
       train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
       test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)
       split_idx = {'train': train_idx, 'test': test_idx}

    # Reset model and optimizer
       model.reset_parameters()
       optimizer = torch.optim.Adam(model.parameters(), lr=lr)

       best_train_rocauc = 0
       loss_history =[]
    # Training loop for the current fold
       for epoch in range(1, 1 + epochs):
           loss = train(model, data, train_idx, optimizer)
           if epoch % eval_steps == 0:
              train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)
              train_rocauc = train_metrics[4]
              train_loss = F.cross_entropy(model(data.x, data.adj_t)[train_idx], data.y[train_idx]).item()
              loss_history.append(loss)

           if len(loss_history) > 10:
                  loss_history.pop(0)

           if len(loss_history) == 10:
                loss_diff = [abs(loss_history[i] - loss_history[i - 1]) for i in range(1, len(loss_history))]
                avg_loss_diff = sum(loss_diff) / len(loss_diff)

                # If average loss difference is very small, stop training
                if avg_loss_diff < 1e-3:  # You can adjust the threshold as needed
                    print("Loss discrepancy stayed constant for 10 epochs, stopping training.")
                    break

           if epoch % log_steps == 0:
                 train_acc, train_prec, train_rec, train_f1, train_rocauc, train_auprc = train_metrics
                 test_acc, test_prec, test_rec, test_f1, test_rocauc, test_auprc = test_metrics

                 print(f'Epoch: {epoch:02d}, '
                       f'Loss: {loss:.4f}, '
                       f'Train ROC AUC: {100 * train_rocauc:.2f}%, '
                       f'Test ROC AUC: {100 * test_rocauc:.2f}%')

Fold 2/5
Epoch: 10, Loss: 0.6471, Train ROC AUC: 80.86%, Test ROC AUC: 81.70%


/tmp/ipykernel_4844/2207014343.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
/tmp/ipykernel_4844/2207014343.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)


Epoch: 20, Loss: 0.5759, Train ROC AUC: 81.79%, Test ROC AUC: 82.64%
Epoch: 30, Loss: 0.5637, Train ROC AUC: 82.13%, Test ROC AUC: 82.94%
Epoch: 40, Loss: 0.5396, Train ROC AUC: 83.39%, Test ROC AUC: 84.33%
Epoch: 50, Loss: 0.4998, Train ROC AUC: 84.79%, Test ROC AUC: 85.81%
Epoch: 60, Loss: 0.4684, Train ROC AUC: 84.85%, Test ROC AUC: 85.73%
Epoch: 70, Loss: 0.4520, Train ROC AUC: 84.88%, Test ROC AUC: 85.86%
Epoch: 80, Loss: 0.4285, Train ROC AUC: 85.77%, Test ROC AUC: 86.73%
Epoch: 90, Loss: 0.4228, Train ROC AUC: 85.37%, Test ROC AUC: 86.31%
Epoch: 100, Loss: 0.4155, Train ROC AUC: 85.66%, Test ROC AUC: 86.70%
Epoch: 110, Loss: 0.4071, Train ROC AUC: 86.48%, Test ROC AUC: 87.22%
Epoch: 120, Loss: 0.4048, Train ROC AUC: 86.44%, Test ROC AUC: 87.13%
Epoch: 130, Loss: 0.4105, Train ROC AUC: 86.30%, Test ROC AUC: 86.75%
Epoch: 140, Loss: 0.4122, Train ROC AUC: 85.34%, Test ROC AUC: 85.79%
Epoch: 150, Loss: 0.3857, Train ROC AUC: 86.58%, Test ROC AUC: 86.50%
Epoch: 160, Loss: 0.3714, Tr

In [ ]:
'''model = GAT(1024,256,64, 2, 2, 0.5).to(device)
model.load_state_dict(torch.load('/content/best_model_fold_1.pt'))'''

"model = GAT(1024,256,64, 2, 2, 0.5).to(device)\nmodel.load_state_dict(torch.load('/content/best_model_fold_1.pt'))"

In [ ]:
torch.save(model.state_dict(), '/content/gdrive/MyDrive/Model/GCN(400, 1024, 2, 4, 0.3).pt')

In [ ]:
df_features.shape

(4819, 1026)

In [ ]:
putative_data = df_features.iloc[4819:,0]

In [ ]:
putative_num = len(G.nodes) - num_nodes
pu_indices = np.arange(4819,len(G.nodes()))
putative_idx  = torch.tensor(pu_indices, dtype=torch.long)

In [ ]:
out = model(data.x, data.adj_t)[putative_idx]

In [ ]:
probabilities = torch.softmax(out, dim=1)  # Shape: [batch_size, 2]

#GCN_PB IID (Not Having Isolated Nodes)

##Load Libaries

In [ ]:
!pip install ogb
!pip install torch_geometric

In [ ]:
pip install torch

##GCN Model Layers

In [ ]:
from torch._C import *
import torch
print(torch.__version__)

2.11.0+cu128


In [ ]:
'''import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_classes, num_layers, dropout):
        super(GAT, self).__init__()
        self.convs = torch.nn.ModuleList()

        # GAT layers
        self.convs.append(GATConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels, hidden_channels))
        self.convs.append(GATConv(hidden_channels, out_channels))

        # MLP head layers (added between GAT output and final classifier)
        head_hidden = max(out_channels // 2, 1)  # example: shrink hidden dim
        self.head = torch.nn.Sequential(
            torch.nn.Linear(out_channels, head_hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(head_hidden, out_channels),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout)
        )

        # Final classification layer
        self.fc = torch.nn.Linear(out_channels, num_classes)

        self.dropout = dropout

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for layer in self.head:
            if hasattr(layer, 'reset_parameters'):
                layer.reset_parameters()
        self.fc.reset_parameters()

    def forward(self, x, adj_t):
        # GAT feature extraction
        for conv in self.convs:
            x = conv(x, adj_t)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Pass through MLP head
        x = self.head(x)

        # Final prediction
        x = self.fc(x)
        return x'''

"import numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch_geometric.nn import GATConv\nfrom sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score\nfrom sklearn.metrics import precision_recall_curve, auc\nfrom sklearn.preprocessing import label_binarize\n\nclass GAT(torch.nn.Module):\n    def __init__(self, in_channels, hidden_channels, out_channels,\n                 num_classes, num_layers, dropout):\n        super(GAT, self).__init__()\n        self.convs = torch.nn.ModuleList()\n\n        # GAT layers\n        self.convs.append(GATConv(in_channels, hidden_channels))\n        for _ in range(num_layers - 2):\n            self.convs.append(GATConv(hidden_channels, hidden_channels))\n        self.convs.append(GATConv(hidden_channels, out_channels))\n\n        # MLP head layers (added between GAT output and final classifier)\n        head_hidden = max(out_channels // 2, 1)  # example: shrink hidden dim\n        self.head = torc

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINConv
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize

# GIN Model Implementation
class GIN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_classes, num_layers, dropout):
        super(GIN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.dropout = dropout

        # GIN requires an MLP for each GINConv layer
        # First layer
        mlp1 = nn.Sequential(
            nn.Linear(in_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )
        self.convs.append(GINConv(mlp1))

        # Intermediate layers
        for _ in range(num_layers - 2):
            mlp = nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels),
                nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels)
            )
            self.convs.append(GINConv(mlp))

        # Final layer
        mlp_out = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, out_channels)
        )
        self.convs.append(GINConv(mlp_out))

        self.fc = nn.Linear(out_channels, num_classes)

    def forward(self, x, adj_t):
        for conv in self.convs:
            x = conv(x, adj_t)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.fc(x)
        return x

def train(model, data, train_idx, optimizer, weights=None):
    model.train()
    criterion = torch.nn.CrossEntropyLoss(weight=weights)
    optimizer.zero_grad()
    out = model(data.x, data.adj_t)[train_idx]
    loss = criterion(out, data.y[train_idx])
    loss.backward()
    optimizer.step()
    return loss.item()

def to_one_hot(y, num_classes):
    y_one_hot = torch.zeros(y.size(0), num_classes).to(y.device)
    y_one_hot.scatter_(1, y.view(-1, 1), 1)
    return y_one_hot

@torch.no_grad()
def test(model, data, split_idx, num_classes):
    model.eval()
    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    def compute_metrics(y_true_indices, probs):
        y_true_labels = y_true_indices.cpu().numpy()
        y_pred_labels = probs.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)

        # AUC Metrics
        y_true_bin = label_binarize(y_true_labels, classes=[0, 1])
        rocauc = roc_auc_score(y_true_bin, probs.cpu().numpy(), multi_class='ovr')
        precision, recall, _ = precision_recall_curve(y_true_bin[:, 1], probs[:, 1].cpu().numpy())
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(data.y[split_idx['train']], y_probs[split_idx['train']])
    valid_metrics = compute_metrics(data.y[split_idx['valid']], y_probs[split_idx['valid']])
    test_metrics = compute_metrics(data.y[split_idx['test']], y_probs[split_idx['test']])

    return train_metrics, valid_metrics, test_metrics

In [ ]:
'''import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINconv
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,num_classes,num_layers, dropout):
        super(GCN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.fc = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels))
        self.fc.append(torch.nn.Linear(out_channels,num_classes))
        self.dropout = dropout


    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()

    def forward(self, x, adj_t):
        for conv in self.convs:
            x = conv(x, adj_t)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        #second to last layer i use this when i wanna apply dropout after fully connected layer
        for fc in self.fc[:-1]:
            x = fc(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.fc[-1](x)
        return x

def train(model, data, train_idx, optimizer):
      model.train()
      criterion = torch.nn.CrossEntropyLoss(weight=weights)
      optimizer.zero_grad()
      out = model(data.x, data.adj_t)[train_idx]
      loss = criterion(out, data.y[train_idx])
      loss.backward()
      optimizer.step()
      return loss.item()

def to_one_hot(y, num_classes):
    y_one_hot = torch.zeros(y.size(0), num_classes).to(y.device)
    y_one_hot.scatter_(1, y.view(-1, 1), 1)
    return y_one_hot

@torch.no_grad()
def test(model, data, split_idx, num_classes):
    model.eval()

    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    y_true_train = to_one_hot(data.y[split_idx['train']], num_classes)
    y_true_valid = to_one_hot(data.y[split_idx['valid']], num_classes)
    y_true_test = to_one_hot(data.y[split_idx['test']], num_classes)

    def compute_metrics(y_true, y_probs):
        y_pred_labels = y_probs.argmax(dim=1).cpu().numpy()
        y_true_labels = y_true.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)
        rocauc = roc_auc_score(y_true.cpu(), y_probs.cpu())
        #aucprc=0.6
        y_true_cpu = y_true.cpu()
        y_true_binarized = label_binarize(y_true_cpu, classes=[0, 1])
        y_probs_cpu = y_probs.cpu().numpy()
        precision, recall, _ = precision_recall_curve(y_true_binarized[:, 1], [score[1] for score in y_probs_cpu])
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(y_true_train, y_probs[split_idx['train']])
    valid_metrics = compute_metrics(y_true_valid, y_probs[split_idx['valid']])
    test_metrics = compute_metrics(y_true_test, y_probs[split_idx['test']])

    return train_metrics, valid_metrics, test_metrics

@torch.no_grad()
def test_kfold(model, data, split_idx, num_classes):
    model.eval()

    y_probs = model(data.x, data.adj_t)
    y_probs = torch.softmax(y_probs, dim=1)

    y_true_train = to_one_hot(data.y[split_idx['train']], num_classes)
    #y_true_valid = to_one_hot(data.y[split_idx['valid']], num_classes)
    y_true_test = to_one_hot(data.y[split_idx['test']], num_classes)

    def compute_metrics(y_true, y_probs):
        y_pred_labels = y_probs.argmax(dim=1).cpu().numpy()
        y_true_labels = y_true.argmax(dim=1).cpu().numpy()

        acc = accuracy_score(y_true_labels, y_pred_labels)
        prec = precision_score(y_true_labels, y_pred_labels, zero_division=0)
        rec = recall_score(y_true_labels, y_pred_labels, zero_division=0)
        f1 = f1_score(y_true_labels, y_pred_labels, zero_division=0)
        rocauc = roc_auc_score(y_true.cpu(), y_probs.cpu())
        #aucprc=0.6
        y_true_cpu = y_true.cpu()
        y_true_binarized = label_binarize(y_true_cpu, classes=[0, 1])
        y_probs_cpu = y_probs.cpu().numpy()
        precision, recall, _ = precision_recall_curve(y_true_binarized[:, 1], [score[1] for score in y_probs_cpu])
        auprc = auc(recall, precision)

        return [acc, prec, rec, f1, rocauc, auprc]

    train_metrics = compute_metrics(y_true_train, y_probs[split_idx['train']])
    #valid_metrics = compute_metrics(y_true_valid, y_probs[split_idx['valid']])
    test_metrics = compute_metrics(y_true_test, y_probs[split_idx['test']])

    return train_metrics, test_metrics'''

"import numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch_geometric.data import Data\nfrom torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINconv\nfrom sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score\nfrom sklearn.metrics import precision_recall_curve, auc\nfrom sklearn.preprocessing import label_binarize\nimport numpy as np\n\nclass GCN(torch.nn.Module):\n    def __init__(self, in_channels, hidden_channels, out_channels,num_classes,num_layers, dropout):\n        super(GCN, self).__init__()\n        self.convs = torch.nn.ModuleList()\n        self.fc = torch.nn.ModuleList()\n        self.convs.append(GCNConv(in_channels, hidden_channels))\n        for _ in range(num_layers - 2):\n            self.convs.append(GCNConv(hidden_channels, hidden_channels))\n        self.convs.append(GCNConv(hidden_channels, out_channels))\n        self.fc.append(torch.nn.Linear(out_channels,num_classes))\n        self.dropout = dropou

##Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import pandas as pd
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/embeddings_mean_protbert_3k.csv')
df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/protT5_embeddings.csv')
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/New_embeddings_protbert_whole_nodes.csv')
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/New_embeddings_protbert_whole.csv')
#df_features = df_features.iloc[:19550,:]
#df_features.iloc[:,1:-1] = np.random.uniform(low=-3, high=6, size=(5609, 1024)) # replace with random

df_labels = df_features .iloc[:4819, [0, -1]]

In [ ]:
df_labels.shape

In [ ]:
# Convert both ID columns to strings (example)
df_labels['Id'] = df_labels['Id'].astype(str)
df_features['Id'] = df_features['Id'].astype(str)

In [ ]:
order_dict = {value: index for index, value in enumerate(list(df_labels['Id']))}

def sorting_key(value):
    return (order_dict.get(value, float('inf')), value)

df_features = df_features.sort_values(by='Id', key=lambda x: x.map(sorting_key))

In [ ]:
df_features.shape

In [ ]:
import networkx as nx

graph_data = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/new_ppi_edges_iid.csv')
#graph_data = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/new_ppi_edges_iid_whole.csv')
#graph_data = pd.read_csv('/content/gdrive/MyDrive/Data/PPI-edges-STRING.csv')

id_list = list(df_features['Id'])

G = nx.Graph()
for id1,id2 in zip(list(graph_data.iloc[:,0]),list(graph_data.iloc[:,1])):
  if((str(id1) in id_list) and (str(id2) in id_list)):
    G.add_edge(id1,id2)

In [ ]:
len(G.edges)

In [ ]:
len(G.nodes())

In [ ]:
df_features = df_features[df_features['Id'].isin(list(G.nodes()))]

In [ ]:
last_col = df_features.columns[-1]
count = (df_features[last_col] != -1).sum()
print("Number of rows where label is not -1:", count)

In [ ]:
adj_sparse = nx.adjacency_matrix(G, nodelist=list(df_features.iloc[:,0]))
node_labels = np.array(df_features.iloc[:,-1])
from scipy.sparse import coo_matrix
adj_matrix = coo_matrix(adj_sparse).todense()

In [ ]:
import os
import numpy as np
import torch

# Define the number of nodes and features
num_nodes = 3730
num_features = 1024
num_classes = 2

node_features = np.array(df_features.iloc[:,1:-1])
node_names = np.array(df_features.iloc[:,0])

adj_matrix = (adj_matrix + adj_matrix.T) / 2
adj_matrix[adj_matrix < 0.9] = 0  # Sparsify the adjacency matrix

node_features_tensor = torch.from_numpy(node_features).float()
node_labels_tensor = torch.from_numpy(node_labels).long()  # Convert labels to long type

adj_coo = adj_sparse.tocoo()
indices = np.vstack((adj_coo.row, adj_coo.col))
values = adj_sparse.data

indices_tensor = torch.tensor(indices, dtype=torch.long)
values_tensor = torch.tensor(values, dtype=torch.float)

adj_t = torch.sparse_coo_tensor(indices_tensor, values_tensor, adj_sparse.shape).to_sparse_csr()

data = Data(uni_id=node_names ,x=node_features_tensor, adj_t=adj_t, y=node_labels_tensor)

# Assuming your data structure
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Map unique IDs to indices
id_to_idx_map = {id_: idx for idx, id_ in enumerate(data.uni_id)}

# Function to load folds (train.npy and test.npy) from Google Drive
def load_folds(folds_dir):
    folds = []
    for fold in range(1, 6):  # Assuming 5 folds (1 to 5)
        # Define the paths for train.npy and test.npy for each fold
        train_path = os.path.join(folds_dir, f'fold_{fold}_train_ids.csv')
        test_path = os.path.join(folds_dir, f'fold_{fold}_test_ids.csv')

        # Load the train and test files
        train_ids = pd.read_csv(train_path)  # This should load the numpy array of IDs
        test_ids = pd.read_csv(test_path)

        # Map the IDs to indices
        train_idx = torch.tensor([id_to_idx_map[id_] for id_ in list(train_ids.iloc[:,0]) if id_ in id_to_idx_map],dtype=torch.long).to(device)
        #train_idx = torch.tensor([id_to_idx_map[id_] for id_ in list (train_ids.iloc[:,0])], dtype=torch.long).to(device)
        test_idx = torch.tensor([id_to_idx_map[id_] for id_ in list(test_ids.iloc[:,0]) if id_ in id_to_idx_map],dtype=torch.long).to(device)
        #test_idx = torch.tensor([id_to_idx_map[id_] for id_ in list (test_ids.iloc[:,0])], dtype=torch.long).to(device)

        # Append the tuple of train and test indices for this fold
        folds.append((train_idx, test_idx))

    return folds

folds_dir = "/content/gdrive/MyDrive/New_Repo_Data/5folds_IId"  # Update this path to your Google Drive directory
folds = load_folds(folds_dir)

##Run Model

In [ ]:
import os
import numpy as np
import torch
from sklearn.model_selection import KFold
import torch.nn.functional as F

# Hyper-parameters and settings
hidden_channels = 64
out_channels = 16
num_layers = 2
dropout = 0.5
runs = 5
lr = 0.001
epochs = 500
eval_steps = 10
log_steps = 10
weights = torch.tensor([1.5,2.5])
num_classes = 2
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load data and folds
data = data.to(device)
weights = weights.to(device)
model = GIN(in_channels=data.num_features,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            num_classes=num_classes,
            num_layers=num_layers,
            dropout=dropout).to(device)

# Metrics storage
train_acc_list, train_prec_list, train_rec_list = [], [], []
train_f1_list, train_rocauc_list, train_aucprc_list = [], [], []
test_acc_list, test_prec_list, test_rec_list = [], [], []
test_f1_list, test_rocauc_list, test_aucprc_list = [], [], []

# Main training loop
for fold, (train_idx, test_idx) in enumerate(folds):
    print(f'\n=== Fold {fold + 1}/{len(folds)} ===')
    train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
    test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)
    split_idx = {'train': train_idx, 'test': test_idx}

    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_train_rocauc = 0
    loss_history = []

    for epoch in range(1, epochs + 1):
        loss = train(model, data, train_idx, optimizer)
        loss_history.append(loss)

        # Early stopping check
        if len(loss_history) > 10 and np.std(loss_history[-10:]) < 1e-3:
            print(f"Early stopping at epoch {epoch}")
            break

        # Evaluation
        if epoch % eval_steps == 0 or epoch == epochs:
            train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)
            train_rocauc = train_metrics[4]

            # Save best model
            if train_rocauc > best_train_rocauc:
                best_train_rocauc = train_rocauc
                torch.save(model.state_dict(), f'best_model_fold_{fold}.pt')

    # Final evaluation with best model
    model.load_state_dict(torch.load(f'best_model_fold_{fold}.pt'))
    train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)

    # Store metrics
    for lst, values in zip([train_acc_list, train_prec_list, train_rec_list, train_f1_list, train_rocauc_list, train_aucprc_list],
                           train_metrics):
        lst.append(values)

    for lst, values in zip([test_acc_list, test_prec_list, test_rec_list, test_f1_list, test_rocauc_list, test_aucprc_list],
                           test_metrics):
        lst.append(values)

# Metrics calculations
def calculate_stats(metric_list, name):
    return {
        'mean': np.mean(metric_list),
        'std': np.std(metric_list),
        'var': np.var(metric_list),
        'min': np.min(metric_list),
        'max': np.max(metric_list)
    }

# Generate report
print("\n=== Final Report ===")
print("{:<15} {:<8} {:<8} {:<8} {:<8} {:<8}".format(
    'Metric', 'Mean', 'Std', 'Var', 'Min', 'Max'))

for metric_name, train_list, test_list in [
    ('Accuracy', train_acc_list, test_acc_list),
    ('Precision', train_prec_list, test_prec_list),
    ('Recall', train_rec_list, test_rec_list),
    ('F1', train_f1_list, test_f1_list),
    ('ROC AUC', train_rocauc_list, test_rocauc_list),
    ('PR AUC', train_aucprc_list, test_aucprc_list)
]:
    train_stats = calculate_stats(train_list, 'Train')
    test_stats = calculate_stats(test_list, 'Test')

    print(f"\n**{metric_name}**")
    print("Train:\t{mean:.4f} ± {std:.4f}\t(var: {var:.4f})\t[{min:.4f}-{max:.4f}]".format(**train_stats))
    print("Test:\t{mean:.4f} ± {std:.4f}\t(var: {var:.4f})\t[{min:.4f}-{max:.4f}]".format(**test_stats))

In [ ]:
import os
import numpy as np
import torch
from sklearn.model_selection import KFold
import torch.nn.functional as F

# Hyper-parameters and settings
hidden_channels = 256
out_channels = 64
num_layers = 2
dropout = 0.2
runs = 5
lr = 0.001
epochs = 500
eval_steps = 10
log_steps = 10
weights = torch.tensor([2.0,1.5])
num_classes = 2
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Load data and folds
data = data.to(device)
weights = weights.to(device)
model = GCN(in_channels=data.num_features,
            hidden_channels=hidden_channels,
            out_channels=out_channels,
            num_classes=num_classes,
            num_layers=num_layers,
            dropout=dropout).to(device)

# Metrics storage
train_acc_list, train_prec_list, train_rec_list = [], [], []
train_f1_list, train_rocauc_list, train_aucprc_list = [], [], []
test_acc_list, test_prec_list, test_rec_list = [], [], []
test_f1_list, test_rocauc_list, test_aucprc_list = [], [], []

# Main training loop
for fold, (train_idx, test_idx) in enumerate(folds):
    print(f'\n=== Fold {fold + 1}/{len(folds)} ===')
    train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
    test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)
    split_idx = {'train': train_idx, 'test': test_idx}

    model.reset_parameters()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_train_rocauc = 0
    loss_history = []

    for epoch in range(1, epochs + 1):
        loss = train(model, data, train_idx, optimizer)
        loss_history.append(loss)

        # Early stopping check
        if len(loss_history) > 10 and np.std(loss_history[-10:]) < 1e-3:
            print(f"Early stopping at epoch {epoch}")
            break

        # Evaluation
        if epoch % eval_steps == 0 or epoch == epochs:
            train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)
            train_rocauc = train_metrics[4]

            # Save best model
            if train_rocauc > best_train_rocauc:
                best_train_rocauc = train_rocauc
                torch.save(model.state_dict(), f'best_model_fold_{fold}.pt')

    # Final evaluation with best model
    model.load_state_dict(torch.load(f'best_model_fold_{fold}.pt'))
    train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)

    # Store metrics
    for lst, values in zip([train_acc_list, train_prec_list, train_rec_list, train_f1_list, train_rocauc_list, train_aucprc_list],
                           train_metrics):
        lst.append(values)

    for lst, values in zip([test_acc_list, test_prec_list, test_rec_list, test_f1_list, test_rocauc_list, test_aucprc_list],
                           test_metrics):
        lst.append(values)

    # --- PRINT FOLD METRICS ---
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC', 'PR AUC']
    print(f"\nFold {fold+1} Metrics:")
    print("{:<12}  {:<18} {:<18}".format("Metric", "Train", "Test"))
    for i, name in enumerate(metric_names):
        print("{:<12}  {:<18.4f} {:<18.4f}".format(name, train_metrics[i], test_metrics[i]))
    # --- END PRINT FOLD METRICS ---


# Metrics calculations
def calculate_stats(metric_list, name):
    return {
        'mean': np.mean(metric_list),
        'std': np.std(metric_list),
        'var': np.var(metric_list),
        'min': np.min(metric_list),
        'max': np.max(metric_list)
    }

# Generate report
print("\n=== Final Report ===")
print("{:<15} {:<8} {:<8} {:<8} {:<8} {:<8}".format(
    'Metric', 'Mean', 'Std', 'Var', 'Min', 'Max'))

for metric_name, train_list, test_list in [
    ('Accuracy', train_acc_list, test_acc_list),
    ('Precision', train_prec_list, test_prec_list),
    ('Recall', train_rec_list, test_rec_list),
    ('F1', train_f1_list, test_f1_list),
    ('ROC AUC', train_rocauc_list, test_rocauc_list),
    ('PR AUC', train_aucprc_list, test_aucprc_list)
]:
    train_stats = calculate_stats(train_list, 'Train')
    test_stats = calculate_stats(test_list, 'Test')

    print(f"\n**{metric_name}**")
    print("Train:\t{mean:.4f} ± {std:.4f}\t(var: {var:.4f})\t[{min:.4f}-{max:.4f}]".format(**train_stats))
    print("Test:\t{mean:.4f} ± {std:.4f}\t(var: {var:.4f})\t[{min:.4f}-{max:.4f}]".format(**test_stats))


In [ ]:
for fold, (train_idx, test_idx) in enumerate(folds):
    if fold == 2:
       print(f'Fold {fold + 1}/{len(folds)}')

    # Convert indices to tensors and move to GPU/CPU
       train_idx = torch.tensor(train_idx, dtype=torch.long).to(device)
       test_idx = torch.tensor(test_idx, dtype=torch.long).to(device)
       split_idx = {'train': train_idx, 'test': test_idx}

    # Reset model and optimizer
       model.reset_parameters()
       optimizer = torch.optim.Adam(model.parameters(), lr=lr)

       best_train_rocauc = 0
       loss_history =[]
    # Training loop for the current fold
       for epoch in range(1, 1 + epochs):
           loss = train(model, data, train_idx, optimizer)
           if epoch % eval_steps == 0:
              train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)
              train_rocauc = train_metrics[4]
              train_loss = F.cross_entropy(model(data.x, data.adj_t)[train_idx], data.y[train_idx]).item()
              loss_history.append(loss)

           if len(loss_history) > 10:
                  loss_history.pop(0)

           if len(loss_history) == 10:
                loss_diff = [abs(loss_history[i] - loss_history[i - 1]) for i in range(1, len(loss_history))]
                avg_loss_diff = sum(loss_diff) / len(loss_diff)

                # If average loss difference is very small, stop training
                if avg_loss_diff < 1e-3:  # You can adjust the threshold as needed
                    print("Loss discrepancy stayed constant for 10 epochs, stopping training.")
                    break

           if epoch % log_steps == 0:
                 train_acc, train_prec, train_rec, train_f1, train_rocauc, train_auprc = train_metrics
                 test_acc, test_prec, test_rec, test_f1, test_rocauc, test_auprc = test_metrics

                 print(f'Epoch: {epoch:02d}, '
                       f'Loss: {loss:.4f}, '
                       f'Train ROC AUC: {100 * train_rocauc:.2f}%, '
                       f'Test ROC AUC: {100 * test_rocauc:.2f}%')

In [ ]:
model = GAT(1024,256,64, 2, 2, 0.5).to(device)
model.load_state_dict(torch.load('/content/best_model_fold_1.pt'))

In [ ]:
torch.save(model.state_dict(), '/content/gdrive/MyDrive/Model/GAT(400, 1024, 2, 4, 0.3).pt')

In [ ]:
df_features.shape

In [ ]:
putative_data = df_features.iloc[4819:,0]

In [ ]:
putative_num = len(G.nodes) - num_nodes
pu_indices = np.arange(4819,len(G.nodes()))
putative_idx  = torch.tensor(pu_indices, dtype=torch.long)

In [ ]:
out = model(data.x, data.adj_t)[putative_idx]

In [ ]:
probabilities = torch.softmax(out, dim=1)  # Shape: [batch_size, 2]

#New Putative Data

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
import random
import numpy as np
import torch
import os

seed = 100

# Python random
random.seed(seed)

# Numpy
np.random.seed(seed)

# PyTorch CPU
torch.manual_seed(seed)

# PyTorch GPU
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# For CUDA operations reproducibility
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print(f"Seed set to {seed}")

Seed set to 100


In [ ]:
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/New_embeddings_protbert_whole_nodes.csv')
import pandas as pd
#df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/new_emb_prot.csv')
df_features = pd.read_csv('/content/gdrive/MyDrive/New_Repo_Data/new_emb_protT5.csv')
for node in df_features.iloc[:, 0]:
    if node not in G.nodes():
        G.add_node(node)

In [ ]:
ids = df_features.iloc[:4819,0]
id_to_idx_map = {id_: idx for idx, id_ in enumerate(data.uni_id)}
train_idx = torch.tensor([id_to_idx_map[id_] for id_ in list (ids.iloc[:])], dtype=torch.long).to(device)

In [ ]:
split_idx['train'] = train_idx
split_idx['test'] = split_idx['train']

In [ ]:
model = GIN(1024,256, 64, 2, 2, 0.1).to(device)

In [ ]:
model.reset_parameters()
lr = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_train_rocauc = 0
epochs = 3000
    # Training loop for the current fold
loss_history =[]
for epoch in range(1, 1 + epochs):
           loss = train(model, data, train_idx, optimizer)
           if epoch % eval_steps == 0:
              train_metrics, test_metrics = test_kfold(model, data, split_idx, num_classes)
              train_rocauc = train_metrics[4]
              train_loss = F.cross_entropy(model(data.x, data.adj_t)[train_idx], data.y[train_idx]).item()
              loss_history.append(loss)

           if len(loss_history) > 10:
                  loss_history.pop(0)

           if len(loss_history) == 10:
                loss_diff = [abs(loss_history[i] - loss_history[i - 1]) for i in range(1, len(loss_history))]
                avg_loss_diff = sum(loss_diff) / len(loss_diff)

                # If average loss difference is very small, stop training
                if avg_loss_diff < 1e-3:  # You can adjust the threshold as needed
                    print("Loss discrepancy stayed constant for 10 epochs, stopping training.")
                    break

           best_train_rocauc = 0
           if epoch % log_steps == 0:
                 train_acc, train_prec, train_rec, train_f1, train_rocauc, train_auprc = train_metrics
                 #test_acc, test_prec, test_rec, test_f1, test_rocauc, test_auprc = test_metrics
                 train_rocauc = train_metrics[4]

        # Save the best model for this fold
                 if train_rocauc > best_train_rocauc:
                       best_train_rocauc = train_rocauc
                       torch.save(model.state_dict(), '/content/best_model_all_data.pt')
                       print(f"Saved new best model for fold {fold} with Train ROC AUC: {train_rocauc:.4f}")

                 print(f'Epoch: {epoch:02d}, '
                       f'Train ROC AUC: {100 * train_rocauc:.2f}%, '
                       f'Loss: {loss:.4f}, ')

Saved new best model for fold 4 with Train ROC AUC: 0.8000
Epoch: 10, Train ROC AUC: 80.00%, Loss: 4.7519, 
Saved new best model for fold 4 with Train ROC AUC: 0.7985
Epoch: 20, Train ROC AUC: 79.85%, Loss: 1.4926, 
Saved new best model for fold 4 with Train ROC AUC: 0.8271
Epoch: 30, Train ROC AUC: 82.71%, Loss: 0.6399, 
Saved new best model for fold 4 with Train ROC AUC: 0.8293
Epoch: 40, Train ROC AUC: 82.93%, Loss: 0.5558, 
Saved new best model for fold 4 with Train ROC AUC: 0.8540
Epoch: 50, Train ROC AUC: 85.40%, Loss: 0.5145, 
Saved new best model for fold 4 with Train ROC AUC: 0.8539
Epoch: 60, Train ROC AUC: 85.39%, Loss: 0.4842, 
Saved new best model for fold 4 with Train ROC AUC: 0.8550
Epoch: 70, Train ROC AUC: 85.50%, Loss: 0.4533, 
Saved new best model for fold 4 with Train ROC AUC: 0.8543
Epoch: 80, Train ROC AUC: 85.43%, Loss: 0.4317, 
Saved new best model for fold 4 with Train ROC AUC: 0.8641
Epoch: 90, Train ROC AUC: 86.41%, Loss: 0.4241, 
Saved new best model for fol

In [ ]:
print(model)
model.load_state_dict(torch.load('/content/best_model_all_data.pt'))

GIN(
  (convs): ModuleList(
    (0): GINConv(nn=Sequential(
      (0): Linear(in_features=1024, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=256, bias=True)
    ))
    (1): GINConv(nn=Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=64, bias=True)
    ))
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


<All keys matched successfully>

In [ ]:
#putative_data = df_features.iloc[4819:,0]

In [ ]:
adj_sparse = nx.adjacency_matrix(G, nodelist=list(df_features.iloc[:,0]))
node_labels = np.array(df_features.iloc[:,-1])
from scipy.sparse import coo_matrix
adj_matrix = coo_matrix(adj_sparse).todense()

In [ ]:
len(adj_matrix)

6421

In [ ]:
import os
import numpy as np
import torch

# Define the number of nodes and features
num_nodes = 4819
num_features = 1024
num_classes = 2

node_features = np.array(df_features.iloc[:,1:-1])
node_names = np.array(df_features.iloc[:,0])

adj_matrix = (adj_matrix + adj_matrix.T) / 2
adj_matrix[adj_matrix < 0.9] = 0  # Sparsify the adjacency matrix

node_features_tensor = torch.from_numpy(node_features).float()
node_labels_tensor = torch.from_numpy(node_labels).long()  # Convert labels to long type

adj_coo = adj_sparse.tocoo()
indices = np.vstack((adj_coo.row, adj_coo.col))
values = adj_sparse.data

indices_tensor = torch.tensor(indices, dtype=torch.long)
values_tensor = torch.tensor(values, dtype=torch.float)

adj_t = torch.sparse_coo_tensor(indices_tensor, values_tensor, adj_sparse.shape).to_sparse_csr()

data = Data(uni_id=node_names ,x=node_features_tensor, adj_t=adj_t, y=node_labels_tensor)

# Assuming your data structure
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Map unique IDs to indices
id_to_idx_map = {id_: idx for idx, id_ in enumerate(data.uni_id)}


In [ ]:
putative_idx = np.arange(4819, len(G.nodes()))

In [ ]:
model = model.to(device)
data = data.to(device)

In [ ]:
data

Data(x=[6421, 1024], y=[6421], uni_id=[6421], adj_t=[6421, 6421])

In [ ]:

len(putative_idx )

1602

In [ ]:
len(data.adj_t)

6421

In [ ]:
out = model(data.x, data.adj_t)[putative_idx]

In [ ]:
#binary_labels = (probabilities[:, 1] > 0.5).long()
#binary_labels.sum()

In [ ]:
probabilities = torch.softmax(out, dim=1)  # Shape: [batch_size, 2]
binary_labels = (probabilities[:, 1] >= 0.5).long()

In [ ]:
probabilities

tensor([[1.0000e+00, 6.7969e-09],
        [1.0000e+00, 1.1619e-13],
        [1.0000e+00, 1.8016e-09],
        ...,
        [1.0000e+00, 1.2633e-10],
        [1.0000e+00, 1.6046e-38],
        [1.0000e+00, 7.6856e-16]], device='cuda:0', grad_fn=<SoftmaxBackward0>)

In [ ]:
binary_labels.sum()

tensor(353, device='cuda:0')

In [ ]:
#putative_data = pd.read_csv("/content/gdrive/MyDrive/New_Repo_Data/pu_protwv.csv")
putative_data = df_features.iloc[4819:,:]

In [ ]:
tensor_cpu = binary_labels.cpu().tolist()
prob_cpu = probabilities.cpu().tolist()

# Add columns to DataFrame
putative_data['binary_labels'] = tensor_cpu
putative_data['probabilities'] = prob_cpu

/tmp/ipykernel_4844/2373778613.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  putative_data['binary_labels'] = tensor_cpu
/tmp/ipykernel_4844/2373778613.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  putative_data['probabilities'] = prob_cpu


In [ ]:
putative_data

,Id,0,1,2,3,4,5,6,7,8,...,1017,1018,1019,1020,1021,1022,1023,label,binary_labels,probabilities
4819,O15439,0.016681,0.101299,0.018172,0.014516,-0.009038,0.034322,-0.017712,-0.051151,0.068329,...,0.006280,0.045987,-0.023475,-0.017763,0.002150,-0.033880,-0.016459,-1,0,"[1.0, 6.796863072366932e-09]"
4820,Q92887,0.011862,0.008280,0.012406,-0.009327,-0.022311,0.035066,-0.018894,-0.074778,0.024072,...,-0.009642,0.048607,-0.024504,0.006555,-0.007633,0.006680,-0.018570,-1,0,"[1.0, 1.1619059935106396e-13]"
4821,O15438,0.016918,0.054519,0.024995,0.007762,-0.019492,0.052943,-0.036149,-0.046119,0.055735,...,0.009224,0.043726,-0.036384,-0.015097,-0.022825,-0.017019,0.003032,-1,0,"[1.0, 1.8016009795118748e-09]"
4822,O95342,0.014462,0.056066,0.008695,0.006586,-0.001066,0.014299,-0.009921,-0.055854,0.048382,...,-0.023491,0.048562,-0.010666,-0.007631,0.002130,-0.006177,-0.029932,-1,0,"[1.0, 2.3297355364326755e-11]"
4823,P12822,0.069578,0.105807,0.006674,0.024072,-0.034850,0.005985,-0.026982,-0.040183,0.033886,...,0.038490,0.071703,-0.032931,-0.063769,-0.020192,-0.009326,0.047579,-1,0,"[0.7685050964355469, 0.2314949482679367]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6416,P62424,0.021147,0.146240,0.038134,0.022197,0.006862,0.044965,-0.043981,-0.145434,-0.039382,...,-0.001094,0.018555,-0.034859,-0.047900,0.004066,0.006996,0.054624,-1,0,"[1.0, 2.115477949279849e-34]"
6417,P35237,0.027979,0.044708,0.009378,-0.000457,-0.037758,0.055360,-0.065673,-0.068805,0.031906,...,-0.055274,0.002472,-0.033151,-0.004112,-0.034853,0.043832,0.044949,-1,0,"[1.0, 8.464266071893378e-14]"
6418,Q04721,0.031400,0.098824,0.018971,0.027397,-0.011528,-0.050123,0.012835,-0.068938,0.051455,...,-0.041329,0.068709,0.010063,-0.073326,0.008741,-0.001277,0.017254,-1,0,"[1.0, 1.2632876689178119e-10]"
6419,P62979,0.097718,0.149934,-0.022544,0.004944,0.072132,0.051156,0.008510,-0.177364,-0.098987,...,-0.091276,0.042890,-0.078590,-0.050091,-0.068265,0.022194,0.082047,-1,0,"[1.0, 1.6045982850096758e-38]"


In [ ]:
print(data.y.dtype)
print(data.y.shape)
print(torch.unique(data.y, return_counts=True))
print(data.y[:20])

torch.int64
torch.Size([6421])
(tensor([-1,  0,  1], device='cuda:0'), tensor([1602, 1738, 3081], device='cuda:0'))
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       device='cuda:0')


In [ ]:
putative_data.to_csv("result_GAT_prot_whole.csv")

In [ ]:
tensor_cpu = binary_labels.cpu().tolist()
prob_cpu = probabilities.cpu().tolist()
# Ensure the length of the tensor matches the number of rows in the DataFrame
#assert len(putative_data) == len(tensor_cpu), "The length of the tensor must match the number of rows in the DataFrame."
# Filter the DataFrame for samples with label 1
filtered_df = putative_data[[label == 1 for label in tensor_cpu]]


In [ ]:
filtered_df.to_csv("filtered_data.csv")

In [ ]:
seeds = [1, 10, 21, 42, 100, 123, 2024, 999]

results = {}

for seed in seeds:
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    model = GIN(1024,256,64,2,2,0.1).to(device)
    model.reset_parameters()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # training loop خودت اینجا اجرا شود
    # train(model, ...)

    model.eval()
    with torch.no_grad():
        out = model(data.x, data.adj_t)[putative_idx]
        probabilities = torch.softmax(out, dim=1)
        binary_labels = (probabilities[:,1] >= 0.5).long()

    results[seed] = binary_labels.sum().item()

print(results)

best_seed = max(results, key=results.get)
print("Best seed:", best_seed)
print("Number of predicted positives:", results[best_seed])

{1: 1602, 10: 0, 21: 0, 42: 0, 100: 1602, 123: 1602, 2024: 1602, 999: 1602}
Best seed: 1
Number of predicted positives: 1602
